# Microsam Finetune Evaluation

Evaluate the finetuned microsam model on unseen data. Evaluate the result visually.

In [ ]:
%load_ext autoreload
%autoreload 2

## Monoculture hyphae data from Eva

In [ ]:
from image_processing_tools.util.load_files import find_files_by_pattern
from image_processing_tools.image_class.image_container import ImageContainer
from image_processing_tools.image_class.micro_sam_pipeline import load_config
from pathlib import Path
import matplotlib
# matplotlib.use('Agg') # Critical: prevents plotting to screen/RAM

search_path = (
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET161_ASH1Q670_02_CY5, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET155_ASH1Q670_04_CY5, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET147_ECE1Q670_06_CY5, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET146_ASH1Q670_07_CY5, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET146_ASH1Q670_02_CY5, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET145_ASH1Q670_02_CY5, DAPI",
    )

config_path = Path("./micro_sam_config.json")

file_pattern = ("CROP_*.tif",
                "MAX_*",
                "SHARPEST*C1*.tif",
                "*DIC.tif",
                "MAX_CET145*")

config = load_config(config_path)
config['base_input_dir'] = Path("~/A8/Data_Roan/").expanduser()
config['checkpoint_path'] = Path("~/Projects/C_Albicans Thesis Project/7. Data/microsam_finetune/final_model/vit_b_candida").expanduser()
config['decoder_checkpoint_path'] = Path("~/Projects/C_Albicans Thesis Project/7. Data/microsam_finetune/final_model/vit_b_candida_decoder").expanduser()
config['model_type'] = 'vit_b_lm'
config['segmentation_mode'] = 'automatic'

config['preprocessing']['resize_image'] = False
config['preprocessing']['max_dim'] = 1024
config['preprocessing']['quantization'] = "8bit"
config['preprocessing']['correct_DIC_shift'] = [0,0]
config['ndim'] = 2
config['tiling']['tile_shape'] = (512,512)
config['tiling']['halo'] = (64,64)
config['segmentation_3d']['seed_slice'] = 49
config['segmentation_3d']['projection'] = "mask"

# Find the files. The 'files' variable will hold the list of Path objects.
dic_files = []
for p in search_path:
    dic_files.extend(find_files_by_pattern(p, file_pattern[3], verbose=True))

dic_imgs = [ImageContainer(img,config) for img in dic_files]

In [ ]:
dic_imgs

## Run AIS pipeline

In [ ]:
from image_processing_tools.image_class.micro_sam_pipeline import MicroSAMPipeline
# Uncomment to use the original model for comparison
# config['checkpoint_path'] = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_b_lm").expanduser()
# config['decoder_checkpoint_path'] = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_b_lm_decoder").expanduser()

pipeline = MicroSAMPipeline(dic_imgs,config=config)
pipeline.run()

In [ ]:
pipeline.save_results()

## Load Wang's hyphae data

In [ ]:
search_path = (
    "~/A8/Data_Wang/260213_HWPA_DAPI/",
    )

file_pattern = ("CROP_*.tif",
                "MAX_*",
                "SHARPEST*C1*.tif",
                "*DIC.tif",
                "MAX_CET145*")

dic_files_1 = []
for p in search_path:
    dic_files_1.extend(find_files_by_pattern(p, file_pattern[3], verbose=True))

dic_imgs_1 = [ImageContainer(img,config) for img in dic_files_1]

In [ ]:
dic_imgs_1

In [ ]:
# Set to original models for comparison
# config['checkpoint_path'] = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_b_lm").expanduser()
# config['decoder_checkpoint_path'] = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_b_lm_decoder").expanduser()

pipeline_1 = MicroSAMPipeline(dic_imgs_1,config=config)
pipeline_1.run()

In [ ]:
pipeline_1.save_results()

## Load Eva's coculture data

In [ ]:
search_path = (
    "~/A8/Data_Eva/Training data Chuyao/260415_Ca_Co_She3Mutants_Ca922_6h_F12_ECE1_cFOS/CET146_ECE1Q670_cFOSQ570_06_CY5, CFP, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/260415_Ca_Co_She3Mutants_Ca922_6h_F12_ECE1_cFOS/CET147_ECE1Q670_cFOSQ570_05_CY5, CFP, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/260415_Ca_Co_She3Mutants_Ca922_6h_F12_ECE1_cFOS/CET146_ECE1Q670_cFOSQ570_07_CY5, CFP, DAPI",
    )

file_pattern = ("MAX_C3_*.tif",
                "*DIC.tif",
                "MAX_C2_*.tif")

config['preprocessing']['correct_DIC_shift'] = [5,22]

dic_files_2 = []
fluo_files_2 = []
dapi_files_2 = []
for p in search_path:
    dic_files_2.extend(find_files_by_pattern(p, file_pattern[1], verbose=True))
    fluo_files_2.extend(find_files_by_pattern(p, file_pattern[2], verbose=True))
    dapi_files_2.extend(find_files_by_pattern(p, file_pattern[0], verbose=True))

In [ ]:
merged_imgs = [ImageContainer([di,f,da],config) for di,f,da in zip(dic_files_2,fluo_files_2,dapi_files_2)]
merged_imgs

In [ ]:
merged_imgs[0].display()

In [ ]:
from image_processing_tools.image_class.micro_sam_pipeline import MicroSAMPipeline

# Finetuned model
config['checkpoint_path'] = Path("~/Projects/C_Albicans Thesis Project/7. Data/microsam_finetune/final_model/vit_b_candida").expanduser()
config['decoder_checkpoint_path'] = Path("~/Projects/C_Albicans Thesis Project/7. Data/microsam_finetune/final_model/vit_b_candida_decoder").expanduser()

# OG model
config['checkpoint_path'] = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_b_lm").expanduser()
config['decoder_checkpoint_path'] = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_b_lm_decoder").expanduser()

pipeline_2 = MicroSAMPipeline(merged_imgs,config)
pipeline_2.run()
pipeline_2.save_results()

## Roan's coculture data

In [ ]:
search_path = (
    "~/A8/Data_Roan/260612_Cocu_Mono_DyesTest/Cocu_CET155_24_Phal_Cell_Calc_06_CY5, FITC, DAPI",
    )

file_pattern = ("SHARPEST*_C3_*.tif",
                "*DIC.tif",
                "SHARPEST*_C1_*.tif")

config['preprocessing']['correct_DIC_shift'] = [5,22]

dic_files = []
phall_files = []
calco_files = []
for p in search_path:
    dic_files.extend(find_files_by_pattern(p, file_pattern[1], verbose=True))
    phall_files.extend(find_files_by_pattern(p, file_pattern[2], verbose=True))
    calco_files.extend(find_files_by_pattern(p, file_pattern[0], verbose=True))

merged_imgs_1 = [ImageContainer([d,p,c],config) for d,p,c in zip (dic_files, phall_files, calco_files)]

In [ ]:
merged_imgs_1[0].display()

In [ ]:
config['checkpoint_path'] = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_b_lm").expanduser()
config['decoder_checkpoint_path'] = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_b_lm_decoder").expanduser()

pipeline_3 = MicroSAMPipeline(merged_imgs_1,config)
pipeline_3.run()
pipeline_3.save_results()

## Roan's another co-culture set, DIC only

In [ ]:
search_path = (
    "~/A8/Data_Roan/260327_Cocultures_Mutants_Phalloidin/CET155/03/",
    "~/A8/Data_Roan/260327_Cocultures_Mutants_Phalloidin/CET155/04/",
    "~/A8/Data_Roan/260327_Cocultures_Mutants_Phalloidin/CET155/05/",
    )

file_pattern = ("CROP_*.tif",
                "MAX_*",
                "SHARPEST*C1*.tif",
                "*DIC.tif",
                "MAX_CET145*")

dic_files_roan = []
for p in search_path:
    dic_files_roan.extend(find_files_by_pattern(p, file_pattern[3], verbose=True))

dic_imgs_roan = [ImageContainer(img,config) for img in dic_files_roan]

In [ ]:
from image_processing_tools.image_class.micro_sam_pipeline import MicroSAMPipeline

# Finetuned model
config['checkpoint_path'] = Path("~/Projects/C_Albicans Thesis Project/7. Data/microsam_finetune/final_model/vit_b_candida").expanduser()
config['decoder_checkpoint_path'] = Path("~/Projects/C_Albicans Thesis Project/7. Data/microsam_finetune/final_model/vit_b_candida_decoder").expanduser()

# OG model
# config['checkpoint_path'] = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_b_lm").expanduser()
# config['decoder_checkpoint_path'] = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_b_lm_decoder").expanduser()

pipeline_dic = MicroSAMPipeline(dic_imgs_roan,config)
pipeline_dic.run()
pipeline_dic.save_results()